# 🫀 Machine Learning Practicum
## Heart Disease Prediction - Decision Trees, KNN & SVM
### School of Computing | Machine Learning Fundamentals

---

**Dataset:** `heart_disease.csv` (800 patients, 13 clinical features)  
**Objective:** Predict whether a patient has heart disease (binary classification)  
**Algorithms:** Decision Tree · K-Nearest Neighbours · Support Vector Machine  
**Skills Practiced:** EDA → Preprocessing → Modelling → Hyperparameter Tuning → Evaluation

---

### 📋 Feature Dictionary

| Feature | Type | Description |
|---------|------|-------------|
| `age` | Numeric | Patient age in years |
| `sex` | Binary | 1 = Male, 0 = Female |
| `cp` | Categorical | Chest pain type (0=typical angina, 1=atypical, 2=non-anginal, 3=asymptomatic) |
| `trestbps` | Numeric | Resting blood pressure (mm Hg) |
| `chol` | Numeric | Serum cholesterol (mg/dl) |
| `fbs` | Binary | Fasting blood sugar > 120 mg/dl (1 = true) |
| `restecg` | Categorical | Resting ECG results (0=normal, 1=ST-T abnormality, 2=LV hypertrophy) |
| `thalach` | Numeric | Maximum heart rate achieved |
| `exang` | Binary | Exercise-induced angina (1 = yes) |
| `oldpeak` | Numeric | ST depression induced by exercise |
| `slope` | Categorical | Slope of peak exercise ST segment (0=upsloping, 1=flat, 2=downsloping) |
| `ca` | Numeric | Number of major vessels coloured by fluoroscopy (0–3) |
| `thal` | Categorical | Thalassemia (1=normal, 2=fixed defect, 3=reversible defect) |
| `target` | Binary | **Heart disease present** (1 = yes, 0 = no) - *prediction target* |

---
> **⚠️ Important:** This is a synthetic educational dataset calibrated to realistic clinical statistics.  
> Do **not** use for actual medical decisions.

## Part 0 - Environment Setup

Run this cell first to install/import all required libraries.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn - preprocessing
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      GridSearchCV, StratifiedKFold,
                                      learning_curve)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Sklearn - algorithms
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Sklearn - evaluation
from sklearn.metrics import (accuracy_score, classification_report,
                               confusion_matrix, roc_auc_score,
                               roc_curve, precision_recall_curve,
                               f1_score, ConfusionMatrixDisplay)

# Styling
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.dpi': 110, 'font.family': 'DejaVu Sans',
                     'axes.spines.top': False, 'axes.spines.right': False})

print("✅ All libraries imported successfully!")
print(f"   NumPy {np.__version__}  |  Pandas {pd.__version__}  |  Seaborn {sns.__version__}")

---
## Part 1 - Load & Inspect the Dataset

> **💡 Hint:** Always start by understanding the *shape*, *data types*, *missing values*, and *class balance* before doing any modelling.

In [ ]:
# ── Load the dataset ─────────────────────────────────────────────────────
df = pd.read_csv('heart_disease.csv')
print(f"Shape: {df.shape}  ({df.shape[0]} patients, {df.shape[1]} features)")
df.head(8)

In [ ]:
# ── Data types and non-null counts ───────────────────────────────────────
df.info()

In [ ]:
# ── Summary statistics ───────────────────────────────────────────────────
df.describe().round(2)

In [ ]:
# ── Missing values ───────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
mv_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(mv_df[mv_df['Missing Count'] > 0])
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

In [ ]:
# ── Class distribution ───────────────────────────────────────────────────
vc = df['target'].value_counts()
print("Class distribution:")
print(vc)
print(f"\nPositive rate (disease): {vc[1]/len(df):.1%}")

---
## Part 2 - Exploratory Data Analysis (EDA)

> **🎯 Goal:** Understand which features are most predictive of heart disease before building any model.  
> Good EDA saves hours of trial-and-error during modelling.

In [ ]:
# ── 2.1 Distribution of key numeric features by target ─────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
num_feats = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
labels    = ['Age', 'Resting BP', 'Cholesterol', 'Max HR', 'ST Depression', 'Major Vessels']

for ax, feat, lbl in zip(axes.flat, num_feats, labels):
    for val, color, name in [(0, '#3B82F6', 'No Disease'), (1, '#EF4444', 'Disease')]:
        ax.hist(df[df['target']==val][feat].dropna(), bins=22, alpha=0.65,
                color=color, label=name, density=True)
    ax.set_title(lbl, fontweight='bold', fontsize=12)
    ax.set_xlabel(feat); ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.suptitle('Distribution of Numeric Features by Heart Disease Status', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_distributions.png', bbox_inches='tight', dpi=130)
plt.show()
print("\n📊 Observation: thalach (max heart rate) and oldpeak (ST depression)")
print("   show the clearest separation between disease / no-disease groups.")

In [ ]:
# ── 2.2 Correlation heatmap ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('eda_correlation.png', bbox_inches='tight', dpi=130)
plt.show()
print("\n📊 Key correlations with target:")
print(corr['target'].drop('target').sort_values(ascending=False).round(3).to_string())

In [ ]:
# ── 2.3 Categorical feature analysis ─────────────────────────────────────
cat_feats = ['cp', 'sex', 'fbs', 'restecg', 'exang', 'slope', 'thal']
cat_labels = ['Chest Pain Type', 'Sex', 'High Fasting BS', 
              'Resting ECG', 'Exercise Angina', 'ST Slope', 'Thalassemia']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes_flat = axes.flat

for ax, feat, lbl in zip(axes_flat, cat_feats, cat_labels):
    ct = pd.crosstab(df[feat], df['target'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['#3B82F6','#EF4444'], 
            edgecolor='white', width=0.7)
    ax.set_title(lbl, fontweight='bold', fontsize=11)
    ax.set_xlabel(feat); ax.set_ylabel('% within group')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(['No Disease','Disease'], fontsize=8)

axes_flat[-1].set_visible(False)   # hide last empty subplot
plt.suptitle('Disease Rate by Categorical Feature', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_categorical.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# ── 2.4 Boxplots: age × max HR grouped by disease ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, feat, lbl in zip(axes, ['thalach', 'oldpeak'], 
                           ['Max Heart Rate Achieved', 'ST Depression (Oldpeak)']):
    data_plot = [df[df['target']==0][feat].dropna(),
                 df[df['target']==1][feat].dropna()]
    bp = ax.boxplot(data_plot, patch_artist=True, notch=True,
                    medianprops={'color':'white','linewidth':2.5})
    bp['boxes'][0].set_facecolor('#3B82F6')
    bp['boxes'][1].set_facecolor('#EF4444')
    ax.set_xticklabels(['No Disease', 'Disease'], fontsize=12)
    ax.set_ylabel(lbl, fontsize=12)
    ax.set_title(f'{lbl} by Disease Status', fontweight='bold', fontsize=12)

plt.suptitle('Boxplot Analysis - Key Numeric Predictors', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_boxplots.png', bbox_inches='tight', dpi=130)
plt.show()

---
## Part 3 - Preprocessing

> **🔑 Key steps:**
> 1. Handle missing values (imputation)
> 2. Encode categorical variables if needed  
> 3. Split into train / test sets (stratified to preserve class balance)
> 4. Scale features - **essential for KNN and SVM, good practice for Decision Trees too**
>
> **💡 Hint:** Always fit the scaler on training data only, then transform both train and test.

In [ ]:
# ── 3.1 Impute missing values ────────────────────────────────────────────
print("Before imputation:")
print(df[['ca','thal']].isnull().sum())

# Median imputation for numeric columns with missing values
imputer = SimpleImputer(strategy='median')
df_clean = df.copy()
df_clean[['ca','thal']] = imputer.fit_transform(df[['ca','thal']])

print("\nAfter imputation:")
print(df_clean[['ca','thal']].isnull().sum())
print(f"\nDataset shape: {df_clean.shape}")

In [ ]:
# ── 3.2 Separate features and target ────────────────────────────────────
X = df_clean.drop(columns=['target'])
y = df_clean['target']

print(f"Features (X): {X.shape}  |  Target (y): {y.shape}")
print(f"Feature names: {list(X.columns)}")
print(f"Class balance: {y.value_counts().to_dict()}")

In [ ]:
# ── 3.3 Train / Test split (stratified) ─────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

print(f"Training set:  {X_train.shape}")
print(f"Test set:      {X_test.shape}")
print(f"Train class balance: {y_train.value_counts().to_dict()}")
print(f"Test  class balance: {y_test.value_counts().to_dict()}")

In [ ]:
# ── 3.4 Feature scaling (StandardScaler) ────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit ONLY on training data
X_test_sc  = scaler.transform(X_test)         # apply same transform to test

print("Scaled training features - mean ≈ 0, std ≈ 1:")
print(pd.DataFrame(X_train_sc, columns=X.columns).describe().round(3).loc[['mean','std']])

In [ ]:
# ── Helper function: full model evaluation report ────────────────────────
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, scaled=False,
                   cv_folds=5):
    """
    Train model, compute CV accuracy, test accuracy, F1, AUC, and plot
    confusion matrix + ROC curve.
    """
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    # CV accuracy
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv_folds, scoring='f1')
    
    # Metrics
    acc  = accuracy_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred)
    auc  = roc_auc_score(y_te, model.predict_proba(X_te)[:,1])
    
    print(f"{'='*55}")
    print(f"  Model: {name}")
    print(f"{'='*55}")
    print(f"  CV F1 (5-fold):  {cv_scores.mean():.4f}  ± {cv_scores.std():.4f}")
    print(f"  Test Accuracy:   {acc:.4f}")
    print(f"  Test F1-Score:   {f1:.4f}")
    print(f"  Test AUC-ROC:    {auc:.4f}")
    print()
    print(classification_report(y_te, y_pred, 
          target_names=['No Disease','Disease']))

    # Plots
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Confusion matrix
    ConfusionMatrixDisplay.from_predictions(
        y_te, y_pred, display_labels=['No Disease','Disease'],
        cmap='Blues', ax=axes[0], colorbar=False)
    axes[0].set_title(f'Confusion Matrix - {name}', fontweight='bold')
    
    # ROC curve
    fpr, tpr, _ = roc_curve(y_te, model.predict_proba(X_te)[:,1])
    axes[1].plot(fpr, tpr, lw=2.5, color='#4338CA', label=f'AUC = {auc:.3f}')
    axes[1].plot([0,1],[0,1],'--',color='#94A3B8',lw=1.5,label='Random')
    axes[1].fill_between(fpr, tpr, alpha=0.08, color='#4338CA')
    axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'ROC Curve - {name}', fontweight='bold')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(f'results_{name.lower().replace(" ","_")}.png', 
                bbox_inches='tight', dpi=130)
    plt.show()
    
    return {'name': name, 'cv_f1': cv_scores.mean(), 'acc': acc, 
            'f1': f1, 'auc': auc}

---
## Part 4 - Decision Tree Classifier

### 🌳 Background
A Decision Tree recursively splits the feature space by asking YES/NO questions.  
Each split chooses the feature and threshold that maximises **information purity** (using Gini impurity or Entropy).

**Key hyperparameters to tune:**
- `max_depth` - controls tree depth (prevents overfitting)
- `min_samples_split` - minimum samples required to split a node
- `min_samples_leaf` - minimum samples at a leaf
- `criterion` - `'gini'` (CART) or `'entropy'` (ID3/C4.5)

---

### ❓ Question 4.1 - Baseline Decision Tree
Train a Decision Tree with `max_depth=None` (fully grown). Evaluate using the `evaluate_model` helper.
> **Hint:** Use `DecisionTreeClassifier(random_state=42)`. Pass unscaled data (Decision Trees don't need scaling).

In [ ]:
# ✍️ YOUR CODE HERE
# Step 1: Create a DecisionTreeClassifier with max_depth=None and random_state=42
# Step 2: Call evaluate_model(name, model, X_train, y_train, X_test, y_test)
# What do you observe? Is the model overfitting?

dt_baseline = ___(___)
results_dt_base = evaluate_model('DT Baseline', dt_baseline, ...)

### ❓ Question 4.2 - Visualise the Decision Tree
Visualise the first 3 levels of the trained decision tree.
> **Hint:** Use `plot_tree()` from sklearn with `max_depth=3, filled=True, feature_names=X.columns`.

In [ ]:
# ✍️ YOUR CODE HERE
fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(___, max_depth=___, filled=True, feature_names=list(X.columns),
          class_names=['No Disease', 'Disease'], ax=ax, fontsize=9,
          impurity=True, proportion=False)
ax.set_title('Decision Tree - First 3 Levels', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

### ❓ Question 4.3 - Depth vs Accuracy Plot
Plot how training accuracy and test accuracy change as `max_depth` goes from 1 to 20.  
Identify the optimal depth.
> **Hint:** Loop over depths, train a DT, compute both train and test accuracy each time.

In [ ]:
# ✍️ YOUR CODE HERE
depths = range(1, 21)
train_accs, test_accs = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=___, random_state=42)
    dt.fit(X_train, y_train)
    train_accs.append(accuracy_score(___, ___))
    test_accs.append(accuracy_score(___, ___))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_accs, 'o-', label='Train Accuracy', color='#3B82F6', lw=2)
ax.plot(depths, test_accs,  's-', label='Test Accuracy',  color='#EF4444', lw=2)
ax.set_xlabel('Max Depth'); ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree Depth vs Accuracy', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

best_depth = depths[np.argmax(test_accs)]
print(f"Optimal max_depth: {best_depth}  →  Test Accuracy: {max(test_accs):.4f}")

### ❓ Question 4.4 - GridSearchCV Hyperparameter Tuning
Use `GridSearchCV` to find the best combination of `max_depth`, `min_samples_split`, and `criterion`.
> **Hint:** Define a `param_grid` dict, then use `GridSearchCV(DT, param_grid, cv=StratifiedKFold(5), scoring='f1')`

In [ ]:
# ✍️ YOUR CODE HERE
param_grid_dt = {
    'max_depth':         [3, 4, 5, 6, 7, 8, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf':  [1, 2, 4],
    'criterion':         ['gini', 'entropy'],
}

grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=___,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='___',         # use f1
    n_jobs=-1,
    verbose=1
)
grid_dt.fit(___, ___)

print("Best parameters:", grid_dt.best_params_)
print(f"Best CV F1: {grid_dt.best_score_:.4f}")

### ❓ Question 4.5 - Evaluate Tuned Decision Tree
Evaluate the best estimator from GridSearchCV on the test set.

In [ ]:
# ✍️ YOUR CODE HERE
best_dt = grid_dt.best_estimator_
results_dt_tuned = evaluate_model('DT Tuned', best_dt, 
                                   X_train, y_train, X_test, y_test)

### ❓ Question 4.6 - Feature Importance
Plot the top-10 most important features from the tuned Decision Tree.
> **Hint:** Use `best_dt.feature_importances_` and create a horizontal bar chart.

In [ ]:
# ✍️ YOUR CODE HERE
importances = pd.Series(best_dt.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='barh', ax=ax, color='#4338CA', edgecolor='white')
ax.set_title('Top-10 Feature Importances - Tuned Decision Tree', 
             fontweight='bold', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout(); plt.show()

---
## Part 5 - K-Nearest Neighbours (KNN)

### 🔵 Background
KNN classifies a new point by finding its K nearest training examples and using a **majority vote**.  
**Distance-based** - feature scaling is **mandatory**.

**Key hyperparameters:**
- `n_neighbors` (K) - number of neighbours
- `weights` - `'uniform'` (equal vote) or `'distance'` (inverse-distance weighted)
- `metric` - `'euclidean'`, `'manhattan'`, `'minkowski'`
- `p` - Minkowski power (p=1 → Manhattan, p=2 → Euclidean)

---

### ❓ Question 5.1 - Baseline KNN
Train a KNN with K=5 using **scaled** features. Why must you scale for KNN?
> **Hint:** Use `X_train_sc` and `X_test_sc` (from Part 3).

In [ ]:
# ✍️ YOUR CODE HERE
knn_baseline = KNeighborsClassifier(n_neighbors=___, weights='uniform')
results_knn_base = evaluate_model('KNN Baseline (K=5)', knn_baseline,
                                   X_train_sc, y_train, X_test_sc, y_test)

### ❓ Question 5.2 - Choose Optimal K
Plot test F1 score for K = 1 to 40. Find the K with best test F1.
> **Hint:** Use a loop, store F1 scores, plot, call `np.argmax()`.

In [ ]:
# ✍️ YOUR CODE HERE
k_range = range(1, 41)
train_f1_list, test_f1_list = [], []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
    knn.fit(X_train_sc, y_train)
    train_f1_list.append(f1_score(y_train, knn.predict(X_train_sc)))
    test_f1_list.append(f1_score(___, knn.predict(___)))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(k_range, train_f1_list, 'o-', color='#3B82F6', lw=2, label='Train F1')
ax.plot(k_range, test_f1_list,  's-', color='#EF4444', lw=2, label='Test F1')
best_k = k_range[np.argmax(test_f1_list)]
ax.axvline(best_k, linestyle='--', color='#10B981', lw=2,
           label=f'Best K={best_k}')
ax.set_xlabel('K (n_neighbors)'); ax.set_ylabel('F1 Score')
ax.set_title('KNN: K vs F1 Score', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()
print(f"Best K = {best_k}  |  Test F1 = {max(test_f1_list):.4f}")

### ❓ Question 5.3 - GridSearchCV for KNN
Tune `n_neighbors`, `weights`, and `metric` simultaneously using GridSearchCV.
> **Hint:** `Pipeline` is not needed here since data is already scaled, but you MUST use scaled data.

In [ ]:
# ✍️ YOUR CODE HERE
param_grid_knn = {
    'n_neighbors': list(range(1, 31)),
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan'],
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid=___,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',
    n_jobs=___,
)
grid_knn.fit(___, ___)

print("Best parameters:", grid_knn.best_params_)
print(f"Best CV F1: {grid_knn.best_score_:.4f}")

### ❓ Question 5.4 - Evaluate Tuned KNN

In [ ]:
# ✍️ YOUR CODE HERE
best_knn = grid_knn.best_estimator_
results_knn_tuned = evaluate_model('KNN Tuned', best_knn,
                                    X_train_sc, y_train, X_test_sc, y_test)

### ❓ Question 5.5 - Decision Boundary (2 features)
Plot the KNN decision boundary using only `thalach` and `oldpeak` (two most discriminative features).
> **Hint:** Scale only these 2 features, train a KNN, use `meshgrid` to create a grid of predictions, 
> then use `plt.contourf()`.

In [ ]:
# ✍️ YOUR CODE HERE - 2D decision boundary
feat_a, feat_b = 'thalach', 'oldpeak'

# Scale just these two features
sc2 = StandardScaler()
X2_train = sc2.fit_transform(X_train[[feat_a, feat_b]])
X2_test  = sc2.transform(X_test[[feat_a, feat_b]])

knn2d = KNeighborsClassifier(**grid_knn.best_params_)
knn2d.fit(X2_train, y_train)

# Create mesh grid
h = 0.05
x_min, x_max = X2_train[:,0].min()-1, X2_train[:,0].max()+1
y_min, y_max = X2_train[:,1].min()-1, X2_train[:,1].max()+1
xx2, yy2 = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
Z = knn2d.predict(np.c_[xx2.ravel(), yy2.ravel()]).reshape(xx2.shape)

fig, ax = plt.subplots(figsize=(10, 7))
ax.contourf(xx2, yy2, Z, alpha=0.22, cmap='RdYlBu')
ax.scatter(X2_train[:,0], X2_train[:,1], c=y_train,
           cmap='RdYlBu', edgecolors='white', s=30, alpha=0.8, zorder=3)
ax.set_xlabel(f'{feat_a} (scaled)'); ax.set_ylabel(f'{feat_b} (scaled)')
ax.set_title(f'KNN Decision Boundary (K={grid_knn.best_params_["n_neighbors"]})',
             fontweight='bold')
plt.colorbar(plt.cm.ScalarMappable(cmap='RdYlBu'), ax=ax, label='Class')
plt.tight_layout(); plt.show()

---
## Part 6 - Support Vector Machine (SVM)

### ⚡ Background
SVM finds the **maximum-margin hyperplane** that best separates classes.  
Only the **support vectors** (closest points to the boundary) define the hyperplane.

**Key hyperparameters:**
- `C` - regularisation (small C = wide margin, large C = narrow margin)
- `kernel` - `'linear'`, `'rbf'` (Gaussian), `'poly'`, `'sigmoid'`
- `gamma` - RBF bandwidth (`'scale'`, `'auto'`, or numeric)
- `degree` - polynomial kernel degree

**⚠️ Always use feature scaling before SVM.**

---

### ❓ Question 6.1 - Baseline SVM (Linear Kernel)
Train an SVM with a linear kernel. Use `Pipeline` to bundle scaler + SVM.
> **Hint:** `Pipeline([('scaler', StandardScaler()), ('svm', SVC(...))])`

In [ ]:
# ✍️ YOUR CODE HERE
pipe_svm_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='linear', C=1.0, probability=True, random_state=42)),
])

results_svm_linear = evaluate_model('SVM Linear', pipe_svm_linear,
                                     X_train, y_train, X_test, y_test)

### ❓ Question 6.2 - RBF Kernel SVM
Train an SVM with the RBF kernel. Compare to the linear version.
> **Hint:** Change `kernel='rbf'`, add `gamma='scale'`.

In [ ]:
# ✍️ YOUR CODE HERE
pipe_svm_rbf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel=___, C=1.0, gamma=___, probability=True, random_state=42)),
])

results_svm_rbf = evaluate_model('SVM RBF', pipe_svm_rbf,
                                  X_train, y_train, X_test, y_test)

### ❓ Question 6.3 - Effect of C on Margin Width
Plot how training and test F1 change as C varies from 0.001 to 1000 (log scale).
> **Hint:** Use `np.logspace(-3, 3, 25)` for C values. Keep kernel='rbf', gamma='scale'.

In [ ]:
# ✍️ YOUR CODE HERE
C_values = np.logspace(-3, 3, 25)
train_f1_c, test_f1_c = [], []

for C_val in C_values:
    pipe = Pipeline([('sc', StandardScaler()),
                     ('svm', SVC(kernel='rbf', C=___, gamma='scale',
                                 probability=True, random_state=42))])
    pipe.fit(X_train, y_train)
    train_f1_c.append(f1_score(y_train, pipe.predict(X_train)))
    test_f1_c.append(f1_score(y_test,  pipe.predict(X_test)))

fig, ax = plt.subplots(figsize=(11, 5))
ax.semilogx(C_values, train_f1_c, 'o-', color='#3B82F6', lw=2, label='Train F1')
ax.semilogx(C_values, test_f1_c,  's-', color='#EF4444', lw=2, label='Test F1')
ax.set_xlabel('C (log scale)'); ax.set_ylabel('F1 Score')
ax.set_title('SVM: Regularisation Parameter C vs F1', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.4); plt.tight_layout(); plt.show()

### ❓ Question 6.4 - GridSearchCV for SVM (the hardest part!)
Run a comprehensive grid search over `kernel`, `C`, and `gamma`.
> **⚠️ This may take 1–2 minutes.**  
> **Hint:** Use `'svm__C'`, `'svm__kernel'`, `'svm__gamma'` as parameter names (Pipeline prefix).

In [ ]:
# ✍️ YOUR CODE HERE
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(probability=True, random_state=42)),
])

param_grid_svm = {
    'svm__kernel': ['linear', 'rbf', 'poly'],
    'svm__C':      [0.1, 1, 10, 100],
    'svm__gamma':  ['scale', 'auto', 0.01, 0.1],   # only used by rbf/poly
    'svm__degree': [2, 3],                           # only used by poly
}

grid_svm = GridSearchCV(
    pipe_svm,
    param_grid=___,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',
    n_jobs=___,
    verbose=1,
)
grid_svm.fit(___, ___)

print("Best parameters:", grid_svm.best_params_)
print(f"Best CV F1: {grid_svm.best_score_:.4f}")

### ❓ Question 6.5 - Evaluate Tuned SVM

In [ ]:
# ✍️ YOUR CODE HERE
best_svm = grid_svm.best_estimator_
results_svm_tuned = evaluate_model('SVM Tuned', best_svm,
                                    X_train, y_train, X_test, y_test)

### ❓ Question 6.6 - Heatmap: C vs Gamma (RBF only)
Visualise the GridSearchCV results as a heatmap for C vs gamma (RBF kernel only).
> **Hint:** Use `pd.DataFrame(grid_svm.cv_results_)`, filter for `rbf`, pivot, and use `sns.heatmap()`.

In [ ]:
# ✍️ YOUR CODE HERE
cv_results = pd.DataFrame(grid_svm.cv_results_)
rbf_results = cv_results[cv_results['param_svm__kernel'] == 'rbf'].copy()
rbf_results['C']     = rbf_results['param_svm__C'].astype(float)
rbf_results['gamma'] = rbf_results['param_svm__gamma'].astype(str)

pivot = rbf_results.pivot_table(
    index='gamma', columns='C', values='mean_test_score')

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Mean CV F1'})
ax.set_title('GridSearchCV: C vs Gamma (RBF kernel)', fontweight='bold')
plt.tight_layout(); plt.show()

---
## Part 7 - Model Comparison & Final Analysis

### ❓ Question 7.1 - Compare All Models
Create a summary table and bar chart comparing all tuned models.
> **Hint:** Collect all `results_*` dicts into a list, build a DataFrame, and plot.

In [ ]:
# ✍️ YOUR CODE HERE
# Collect results from all sections
all_results = [
    results_dt_base,
    results_dt_tuned,
    results_knn_base,
    results_knn_tuned,
    results_svm_linear,
    results_svm_rbf,
    results_svm_tuned,
]

results_df = pd.DataFrame(all_results).set_index('name')
results_df = results_df.round(4)
print("\n📊 Model Comparison Summary:")
print(results_df.to_string())

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = ['acc', 'f1', 'auc']
titles  = ['Test Accuracy', 'Test F1-Score', 'Test AUC-ROC']
colors  = sns.color_palette('deep', len(results_df))

for ax, metric, title in zip(axes, metrics, titles):
    results_df[metric].plot(kind='bar', ax=ax, color=colors, edgecolor='white',
                             width=0.7, rot=30)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_ylabel(title)
    ax.set_ylim(0.5, 1.05)
    ax.axhline(0.9, linestyle='--', color='gray', lw=1, alpha=0.5)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', dpi=130)
plt.show()

### ❓ Question 7.2 - Learning Curves
Plot learning curves for the best overall model.  
Learning curves show how model performance improves with more training data.
> **Hint:** Use `sklearn.model_selection.learning_curve()`.

In [ ]:
# ✍️ YOUR CODE HERE - identify best model by AUC
best_name = results_df['auc'].idxmax()
print(f"Best model by AUC: {best_name}")

# Retrieve the best model from the grid search objects
model_map = {
    'DT Baseline': DecisionTreeClassifier(random_state=42),
    'DT Tuned':    grid_dt.best_estimator_,
    'KNN Baseline (K=5)': KNeighborsClassifier(n_neighbors=5),
    'KNN Tuned':   grid_knn.best_estimator_,
    'SVM Linear':  pipe_svm_linear,
    'SVM RBF':     pipe_svm_rbf,
    'SVM Tuned':   grid_svm.best_estimator_,
}
best_model = model_map[best_name]

train_sizes, train_scores, cv_scores = learning_curve(
    best_model, X_train, y_train,
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    train_sizes=np.linspace(0.1, 1.0, 15),
    scoring='f1', n_jobs=-1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.fill_between(train_sizes, train_scores.mean(1)-train_scores.std(1),
                train_scores.mean(1)+train_scores.std(1), alpha=0.15, color='#3B82F6')
ax.fill_between(train_sizes, cv_scores.mean(1)-cv_scores.std(1),
                cv_scores.mean(1)+cv_scores.std(1), alpha=0.15, color='#EF4444')
ax.plot(train_sizes, train_scores.mean(1), 'o-', color='#3B82F6', lw=2, label='Train F1')
ax.plot(train_sizes, cv_scores.mean(1),   's-', color='#EF4444', lw=2, label='CV F1')
ax.set_xlabel('Training Set Size'); ax.set_ylabel('F1 Score')
ax.set_title(f'Learning Curve - {best_name}', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

### ❓ Question 7.3 - Precision-Recall Curve
Plot the Precision-Recall curve for the top 3 models.  
This is especially useful for **imbalanced** or **medical** classification problems.
> **Hint:** Use `precision_recall_curve()`. Plot all 3 curves on the same axes.

In [ ]:
# ✍️ YOUR CODE HERE
top3_names = results_df['auc'].nlargest(3).index.tolist()
print(f"Top 3 models: {top3_names}")

fig, ax = plt.subplots(figsize=(9, 6))
colors_pr = ['#4338CA', '#059669', '#D97706']

for name, color in zip(top3_names, colors_pr):
    m = model_map[name]
    Xtr = X_train_sc if 'KNN' in name else X_train
    Xte = X_test_sc  if 'KNN' in name else X_test
    proba = m.predict_proba(Xte)[:,1]
    precision, recall, _ = precision_recall_curve(y_test, proba)
    ax.plot(recall, precision, lw=2.5, color=color, label=name)

ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve - Top 3 Models', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

---
## Part 8 - Reflection Questions

Answer the following questions in the cell below. Write your answers as comments.

1. Which algorithm performed best? Why do you think that is for this dataset?
2. How did hyperparameter tuning improve each model compared to the baseline?
3. For a medical diagnosis task (heart disease), should you optimise for **Precision** or **Recall**? Why?
4. The Decision Tree is highly interpretable - the SVM is not. When would you choose interpretability over accuracy?
5. What additional features or data sources might improve all three models?
6. How would the results change if the dataset had 10× more samples?

In [ ]:
# ✍️ YOUR ANSWERS HERE

# 1. Best algorithm:

# 2. Impact of hyperparameter tuning:

# 3. Precision vs Recall for medical diagnosis:

# 4. Interpretability vs Accuracy trade-off:

# 5. Additional features/data:

# 6. Impact of more data: